# Safety RLHF Fine-Tuning of Ministral 3 3B on PKU-SafeRLHF

This notebook safety-RLHF fine-tunes **Ministral 3 3B** (`mistralai/Ministral-3-3B-Instruct-2512-BF16`) on the [PKU-SafeRLHF](https://huggingface.co/datasets/PKU-Alignment/PKU-SafeRLHF) preference dataset, using true reinforcement learning (PPO) rather than a more efficient offline method like DPO.

**Pipeline:**
1. Load PKU-SafeRLHF and visualize severity/harm-category distributions.
2. Build disjoint pools for reward-model training, PPO rollouts, and before/after evaluation.
3. Train a LoRA reward model on harmlessness preferences (`safer_response_id`).
4. Run PPO RLHF (TRL `PPOTrainer`) with LoRA on the policy and value model.
5. Visualize PPO training curves.
6. Evaluate safety before vs. after fine-tuning with Llama Guard 3 8B.
7. Save a merged, full-weight model to Google Drive for later use in `rag_safety.ipynb`.

**Designed for a single Google Colab A100 (80GB GPU RAM).** LoRA is used throughout so that the policy, reference (via disabled adapters), reward model, and value model never require more than ~15-25GB of GPU memory concurrently.


# Set up environment

## Install packages

Note that you may need to restart the Colab runtime after this cell for the new versions to load.

In [ ]:
!pip install -q -U \
    "transformers>=4.57.0" \
    "trl>=0.25.0" \
    "peft>=0.18.0" \
    "accelerate>=1.2.0" \
    "bitsandbytes>=0.43.1" \
    "datasets>=2.20.0" \
    "mistral-common>=1.8.6" \
    "huggingface_hub>=0.23.0" \
    "pandas>=2.2.0" \
    "numpy>=1.26.0" \
    "matplotlib>=3.8.0" \
    "seaborn>=0.13.0" \
    "tqdm>=4.66.0" \
    "torchao>=0.16.0"

## Check package versions, GPU, and TRL PPO API location

TRL's PPO trainer has moved between `trl` (stable) and `trl.experimental.ppo` across recent releases. This cell verifies which import path is available in the installed version so later cells import from the right place.

In [ ]:
# Code in this block partially generated with Claude
import importlib
from packaging.version import Version

REQUIRED = {
    "torch": "2.3.0",
    "transformers": "4.57.0",
    "trl": "0.25.0",
    "peft": "0.18.0",
    "accelerate": "1.2.0",
    "bitsandbytes": "0.43.1",
    "datasets": "2.20.0",
    "pandas": "2.2.0",
    "numpy": "1.26.0",
    "torchao": "0.16.0",  # peft's LoRA dispatcher raises ImportError on older torchao, even when unused
}

ok = True
for pkg, min_ver in REQUIRED.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "0")
        status = "OK" if Version(ver) >= Version(min_ver) else "OUTDATED"
        if status == "OUTDATED":
            ok = False
        print(f"  {status:8s} {pkg} {ver} (need >={min_ver})")
    except ImportError:
        print(f"  MISSING  {pkg}")
        ok = False

import torch
cuda_ok = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_ok else "none"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if cuda_ok else 0
print(f"\n  GPU: {gpu_name} ({gpu_mem:.1f} GB)")
if not cuda_ok:
    print("  WARNING: CUDA not available — make sure you selected the A100 runtime.")
if gpu_mem < 40:
    print(f"  WARNING: GPU has only {gpu_mem:.1f} GB. This notebook is designed for an 80GB A100.")
if not ok:
    print("\n  Some packages are outdated or missing — re-run the install cell and restart the runtime.")
else:
    print("\n  All packages OK.")

# Locate the PPOTrainer/PPOConfig implementation: newer TRL ships it under
# `trl.experimental.ppo`, older/future stable releases may expose it at top level.
try:
    from trl.experimental.ppo import PPOConfig, PPOTrainer
    print("\n  Using PPOTrainer/PPOConfig from trl.experimental.ppo")
except ImportError:
    from trl import PPOConfig, PPOTrainer
    print("\n  Using PPOTrainer/PPOConfig from trl (stable)")

## Define environment variables and hyperparameters

In [ ]:
# HuggingFace credentials
# Required for Llama Guard (gated model). Run huggingface-cli login, or set HF_TOKEN here.
HF_TOKEN: str = ""   # leave empty if you have already run huggingface-cli login

# Model IDs
MODEL_ID = "mistralai/Ministral-3-3B-Instruct-2512-BF16"   # BF16 checkpoint (not FP8) for clean LoRA training
LLAMA_GUARD_ID = "meta-llama/Llama-Guard-3-8B"

# Dataset settings
DATASET_ID = "PKU-Alignment/PKU-SafeRLHF"
RM_POOL_SIZE = 5_000        # preference pairs for reward-model training/eval
PPO_PROMPT_POOL_SIZE = 2_000  # unique prompts for PPO rollouts
EVAL_HOLDOUT_SIZE = 400     # unique prompts for before/after safety eval
RM_EVAL_FRACTION = 0.1      # fraction of the RM pool held out for RM eval
SHUFFLE_SEED = 42

# LoRA hyperparameters (used for reward model, value model, and PPO policy)
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Reward model training hyperparameters
RM_LEARNING_RATE = 1e-4
RM_NUM_EPOCHS = 1
RM_BATCH_SIZE = 8
RM_MAX_LENGTH = 1024

# PPO hyperparameters
# `PPO_PER_DEVICE_BATCH_SIZE` is the number of full sequences forwarded/backpropped through the
# policy + value model together in a single step -- this is what drives peak GPU memory during
# the PPO update phase (as opposed to the cheaper no-grad rollout/generation phase). Keep this
# small and use `PPO_GRADIENT_ACCUMULATION_STEPS` to reach the same effective batch size
# (local_batch_size = PPO_PER_DEVICE_BATCH_SIZE * PPO_GRADIENT_ACCUMULATION_STEPS) without ever
# holding more than a few sequences' worth of activations/logits in memory at once. Ministral 3's
# 131k-token vocabulary makes the logits tensor for a full batch very large, so this matters more
# here than for typical small-vocab models.
PPO_LEARNING_RATE = 3e-6
PPO_PER_DEVICE_BATCH_SIZE = 4
PPO_GRADIENT_ACCUMULATION_STEPS = 4  # effective/local batch size = 4 * 4 = 16
PPO_MINI_BATCHES = 1
PPO_NUM_EPOCHS = 4
PPO_RESPONSE_LENGTH = 128
PPO_KL_COEF = 0.05
PPO_MISSING_EOS_PENALTY = 1.0
PPO_MAX_PROMPT_LENGTH = 512  # prompts longer than this are dropped before PPO training

# Generation settings for the before/after safety eval
EVAL_MAX_NEW_TOKENS = 256
EVAL_TEMPERATURE = 0.3


# File paths (Google Drive)print(f"RM pool: {RM_POOL_SIZE:,} pairs | PPO prompts: {PPO_PROMPT_POOL_SIZE:,} | Eval holdout: {EVAL_HOLDOUT_SIZE:,}")

DRIVE_ROOT = "/content/drive/MyDrive/CSCI E-222/final_project/rlhf"print(f"Final merged model -> {FINAL_MODEL_PATH}")

RM_CHECKPOINT_PATH = f"{DRIVE_ROOT}/reward_model"print(f"Reward model checkpoint -> {RM_CHECKPOINT_PATH}")

FINAL_MODEL_PATH = f"{DRIVE_ROOT}/ministral_3b_safety_rlhf"print(f"Policy model: {MODEL_ID}")

EVAL_RESULTS_PATH = f"{DRIVE_ROOT}/before_after_eval.csv"

In [ ]:
import gc
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from tqdm.auto import tqdm

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
)
from peft import LoraConfig
from trl import RewardConfig, RewardTrainer

try:
    from trl.experimental.ppo import PPOConfig, PPOTrainer
except ImportError:
    from trl import PPOConfig, PPOTrainer

from google.colab import drive

warnings.filterwarnings("ignore")

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.bfloat16
print(f"Device: {DEVICE}")

def report_gpu_memory(label: str = "") -> None:
    """Print current GPU memory usage. Call after `del model; gc.collect(); torch.cuda.empty_cache()`."""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"GPU memory in use{' (' + label + ')' if label else ''}: {mem:.1f} GB")

In [ ]:
drive.mount('/content/drive')

# Load and explore PKU-SafeRLHF

Load the full `train` split (~73.9k preference pairs) and inspect its structure before building training splits.

In [ ]:
raw_dataset = load_dataset(DATASET_ID, split="train")
print(f"Loaded {len(raw_dataset):,} rows from {DATASET_ID}")
print(raw_dataset)

In [ ]:
# The 19 harm categories used to label each response (see dataset card)
HARM_CATEGORIES = [
    "Endangering National Security",
    "Insulting Behavior",
    "Discriminatory Behavior",
    "Endangering Public Health",
    "Copyright Issues",
    "Violence",
    "Drugs",
    "Privacy Violation",
    "Economic Crime",
    "Mental Manipulation",
    "Human Trafficking",
    "Physical Harm",
    "Sexual Content",
    "Cybercrime",
    "Disrupting Public Order",
    "Environmental Damage",
    "Psychological Harm",
    "White-Collar Crime",
    "Animal Abuse",
]

SEVERITY_LABELS = {0: "None (safe)", 1: "Minor", 2: "Moderate", 3: "Severe"}

df = raw_dataset.to_pandas()
print(f"DataFrame shape: {df.shape}")
df.head(3)

## Summary statistics

In [ ]:
# Code in this block partially generated with Claude
n = len(df)
n_pairs_both_safe = ((df["is_response_0_safe"]) & (df["is_response_1_safe"])).sum()
n_pairs_both_unsafe = ((~df["is_response_0_safe"]) & (~df["is_response_1_safe"])).sum()
n_pairs_mixed = n - n_pairs_both_safe - n_pairs_both_unsafe

n_unsafe_responses = (~df["is_response_0_safe"]).sum() + (~df["is_response_1_safe"]).sum()
n_total_responses = n * 2

agreement = (df["better_response_id"] == df["safer_response_id"]).mean()

print(f"Total preference pairs: {n:,}")
print(f"  Both responses safe:   {n_pairs_both_safe:,} ({n_pairs_both_safe/n:.1%})")
print(f"  Both responses unsafe: {n_pairs_both_unsafe:,} ({n_pairs_both_unsafe/n:.1%})")
print(f"  Mixed (one safe, one unsafe): {n_pairs_mixed:,} ({n_pairs_mixed/n:.1%})")
print(f"\nUnsafe response rate (of {n_total_responses:,} total responses): {n_unsafe_responses/n_total_responses:.1%}")
print(f"\nAgreement between better_response_id (helpfulness) and safer_response_id (harmlessness): {agreement:.1%}")
print("(Disagreement shows the most helpful response is not always the safest.)")

print("\nPrompt source distribution:")
print(df["prompt_source"].value_counts())

In [ ]:
# Severity level distribution, restricted to unsafe responses (severity 0 = safe, not meaningful to compare)
severity_counts = pd.concat(
    [df.loc[~df["is_response_0_safe"], "response_0_severity_level"],
     df.loc[~df["is_response_1_safe"], "response_1_severity_level"]]
).value_counts().sort_index()
severity_counts.index = severity_counts.index.map(lambda x: SEVERITY_LABELS.get(x, str(x)))
print("Severity level distribution (unsafe responses only):")
print(severity_counts)

# Harm category frequency: aggregate booleans across both responses' harm_category struct columns
category_counts = {cat: 0 for cat in HARM_CATEGORIES}
for col in ["response_0_harm_category", "response_1_harm_category"]:
    for cat in HARM_CATEGORIES:
        category_counts[cat] += df[col].apply(lambda d: bool(d.get(cat, False))).sum()

category_series = pd.Series(category_counts).sort_values(ascending=False)
print("\nHarm category frequency (across all responses):")
print(category_series)

## Visualizations

In [ ]:
# Safe vs. unsafe response counts (both responses pooled)
safe_count = n_total_responses - n_unsafe_responses
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    [safe_count, n_unsafe_responses],
    labels=["Safe", "Unsafe"],
    autopct=lambda p: f"{p:.1f}%\n({int(p / 100 * n_total_responses):,})",
    colors=["#5c8ae0", "#e05c5c"],
    startangle=90,
)
ax.set_title("Safe vs. Unsafe Responses (all responses pooled)", fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
# Severity-level distribution bar chart
fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(severity_counts.index, severity_counts.values, color="#e05c5c", zorder=3)
ax.bar_label(bars, fmt="{:,.0f}", fontsize=12)
ax.set_xlabel("Severity Level", fontsize=16)
ax.set_ylabel("Response Count", fontsize=16)
ax.set_title("Severity Level Distribution (Unsafe Responses)", fontsize=20)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
plt.tight_layout()
plt.show()

In [ ]:
# Harm category frequency bar chart (19 categories)
fig, ax = plt.subplots(figsize=(14, 7))
ax.bar(category_series.index, category_series.values, color="#5c8ae0", zorder=3)
ax.set_xlabel("Harm Category", fontsize=16)
ax.set_ylabel("Response Count", fontsize=16)
ax.set_title("Harm Category Frequency (PKU-SafeRLHF)", fontsize=20)
ax.tick_params(axis="x", labelrotation=45, labelsize=11)
for label in ax.get_xticklabels():
    label.set_ha("right")
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
plt.tight_layout()
plt.show()

In [ ]:
# Helpfulness vs. harmlessness agreement bar chart
agree_counts = pd.Series({
    "Agree (helpful = safer)": (df["better_response_id"] == df["safer_response_id"]).sum(),
    "Disagree (helpful ≠ safer)": (df["better_response_id"] != df["safer_response_id"]).sum(),
})
fig, ax = plt.subplots(figsize=(6, 6))
bars = ax.bar(agree_counts.index, agree_counts.values, color=["#5c8ae0", "#e05c5c"], zorder=3)
ax.bar_label(bars, fmt="{:,.0f}", fontsize=12)
ax.set_ylabel("Pair Count", fontsize=16)
ax.set_title("Helpfulness vs. Harmlessness\nPreference Agreement", fontsize=18)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
plt.tight_layout()
plt.show()

# Sanity check: verify Ministral 3 model classes

The published Ministral 3 checkpoints ship a **vision-language wrapper** config (`model_type: "mistral3"`, class `Mistral3ForConditionalGeneration`) with a nested `text_config` for the actual text-only backbone (`model_type: "ministral3"`, class `Ministral3ForCausalLM`). `AutoModelForCausalLM`/`AutoModelForSequenceClassification` only resolve the text-only `ministral3` config directly — they cannot load the `mistral3` wrapper config, which is only registered for image-text-to-text generation.

Since this project only needs the text backbone (no vision), this cell extracts the `Ministral3ForCausalLM` submodule from the VLM wrapper once, saves it to a local checkpoint, and reassigns `MODEL_ID` to that path. Every later cell references the `MODEL_ID` variable, so this fix propagates automatically to the reward model, PPO policy/value models, and the before/after eval.

In [ ]:
import os
from transformers import Mistral3ForConditionalGeneration, Ministral3ForCausalLM

token_kwarg = {"token": HF_TOKEN} if HF_TOKEN else {}

ORIGINAL_MODEL_ID = MODEL_ID  # e.g. mistralai/Ministral-3-3B-Instruct-2512-BF16 (VLM wrapper checkpoint)
TEXT_MODEL_PATH = f"{DRIVE_ROOT}/ministral_3b_text_backbone"

if not os.path.exists(os.path.join(TEXT_MODEL_PATH, "config.json")):
    print(f"Extracting text-only Ministral3 backbone from {ORIGINAL_MODEL_ID} ...")
    _vlm_model = Mistral3ForConditionalGeneration.from_pretrained(
        ORIGINAL_MODEL_ID, dtype=TORCH_DTYPE, **token_kwarg
    )

    _text_only_model = Ministral3ForCausalLM(_vlm_model.config.text_config)
    _text_only_model.model.load_state_dict(_vlm_model.model.language_model.state_dict())
    _text_only_model.lm_head.load_state_dict(_vlm_model.lm_head.state_dict())
    _text_only_model = _text_only_model.to(dtype=TORCH_DTYPE)
    _text_only_model.save_pretrained(TEXT_MODEL_PATH)

    _text_tokenizer = AutoTokenizer.from_pretrained(ORIGINAL_MODEL_ID, **token_kwarg)
    _text_tokenizer.save_pretrained(TEXT_MODEL_PATH)

    del _vlm_model, _text_only_model, _text_tokenizer
    report_gpu_memory("after text-backbone extraction")
else:
    print(f"Found existing extracted text backbone at {TEXT_MODEL_PATH}")

# Reassign MODEL_ID so every downstream cell (RM, PPO policy/value, before/after eval) uses the
# extracted text-only checkpoint instead of the VLM wrapper checkpoint.
MODEL_ID = TEXT_MODEL_PATH
print(f"MODEL_ID reassigned to: {MODEL_ID}")

_smoke_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **token_kwarg)
_smoke_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=TORCH_DTYPE, device_map="auto", **token_kwarg
)
print(f"Resolved model class: {type(_smoke_model).__name__}")

_smoke_messages = [{"role": "user", "content": "Say hello in one short sentence."}]
_smoke_inputs = _smoke_tokenizer.apply_chat_template(
    _smoke_messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(DEVICE)
with torch.no_grad():
    _smoke_out = _smoke_model.generate(**_smoke_inputs, max_new_tokens=30, do_sample=False)
_smoke_decoded = _smoke_tokenizer.decode(
    _smoke_out[0][_smoke_inputs["input_ids"].shape[-1]:], skip_special_tokens=True
)
print("Smoke test output:", _smoke_decoded)

del _smoke_model
report_gpu_memory("after smoke test cleanup")

# Build reward-model, PPO, and evaluation splits

Construct `chosen`/`rejected` preference pairs from `safer_response_id` (harmlessness only, ignoring `better_response_id`), flag safety-discriminative pairs, then partition the dataset into three disjoint pools:
- **RM pool**: preference pairs for reward-model training/eval, oversampled toward safety-discriminative pairs.
- **PPO prompt pool**: unique prompts (no responses) for PPO rollouts.
- **Eval holdout pool**: unique prompts held out for the before/after safety evaluation.

In [ ]:
# Code in this block partially generated with Claude
rng = np.random.default_rng(SHUFFLE_SEED)

# Drop rows with an empty/whitespace-only response: Ministral 3's mistral_common tokenizer
# raises TokenizerException on empty assistant messages, and a small number of PKU-SafeRLHF
# rows have blank response_0/response_1 strings.
_n_before = len(df)
df = df.loc[df["response_0"].str.strip().astype(bool) & df["response_1"].str.strip().astype(bool)].reset_index(drop=True)
print(f"Dropped {_n_before - len(df):,} rows with an empty response ({len(df):,} remaining)")

# Flag safety-discriminative pairs: responses disagree on safety, or differ in severity level
df["is_discriminative"] = (
    (df["is_response_0_safe"] != df["is_response_1_safe"])
    | (df["response_0_severity_level"] != df["response_1_severity_level"])
)
print(f"Discriminative pairs: {df['is_discriminative'].sum():,} / {len(df):,} ({df['is_discriminative'].mean():.1%})")

# Assign disjoint prompt pools first, so RM/PPO/eval never share a prompt.
unique_prompts = df["prompt"].unique()
rng.shuffle(unique_prompts)

eval_prompts = set(unique_prompts[:EVAL_HOLDOUT_SIZE])
ppo_prompts = set(unique_prompts[EVAL_HOLDOUT_SIZE:EVAL_HOLDOUT_SIZE + PPO_PROMPT_POOL_SIZE])
rm_candidate_mask = ~df["prompt"].isin(eval_prompts | ppo_prompts)
rm_candidates = df.loc[rm_candidate_mask].copy()

print(f"\nUnique prompts total: {len(unique_prompts):,}")
print(f"Eval holdout prompts: {len(eval_prompts):,}")
print(f"PPO prompt pool: {len(ppo_prompts):,}")
print(f"RM candidate pairs (remaining prompts): {len(rm_candidates):,}")

In [ ]:
# Build the RM pool: take all discriminative candidates first (capped at RM_POOL_SIZE),
# then fill any remainder with random non-discriminative candidates for balance.
discriminative = rm_candidates.loc[rm_candidates["is_discriminative"]]
non_discriminative = rm_candidates.loc[~rm_candidates["is_discriminative"]]

n_discriminative_take = min(len(discriminative), RM_POOL_SIZE)
rm_pool = discriminative.sample(n=n_discriminative_take, random_state=SHUFFLE_SEED)

n_remaining = RM_POOL_SIZE - n_discriminative_take
if n_remaining > 0:
    fill = non_discriminative.sample(n=min(n_remaining, len(non_discriminative)), random_state=SHUFFLE_SEED)
    rm_pool = pd.concat([rm_pool, fill], ignore_index=True)

rm_pool = rm_pool.sample(frac=1.0, random_state=SHUFFLE_SEED).reset_index(drop=True)
print(f"RM pool size: {len(rm_pool):,} ({rm_pool['is_discriminative'].mean():.1%} discriminative)")

In [ ]:
from datasets import Dataset

def build_preference_pair(row: pd.Series) -> dict:
    """Build a conversational chosen/rejected pair using safer_response_id (harmlessness).

    `chat_template_kwargs.continue_final_message=True` is required for Ministral 3's
    mistral_common tokenizer, which otherwise rejects conversations ending on an
    assistant turn (it validates as if preparing a prompt for live generation).
    """
    responses = [row["response_0"], row["response_1"]]
    safer_idx = int(row["safer_response_id"])
    chosen_response = responses[safer_idx]
    rejected_response = responses[1 - safer_idx]
    return {
        "prompt": [{"role": "user", "content": row["prompt"]}],
        "chosen": [{"role": "assistant", "content": chosen_response}],
        "rejected": [{"role": "assistant", "content": rejected_response}],
        "chat_template_kwargs": {"continue_final_message": True},
    }

rm_pairs = [build_preference_pair(row) for _, row in rm_pool.iterrows()]
rm_dataset_full = Dataset.from_list(rm_pairs)

rm_split = rm_dataset_full.train_test_split(test_size=RM_EVAL_FRACTION, seed=SHUFFLE_SEED)
rm_train_dataset = rm_split["train"]
rm_eval_dataset = rm_split["test"]

print(f"RM train: {len(rm_train_dataset):,} | RM eval: {len(rm_eval_dataset):,}")
print(rm_train_dataset[0])

In [ ]:
# Build PPO prompt pool and eval holdout pool as plain prompt lists (deduplicated, one row per unique prompt)
ppo_prompt_list = sorted(ppo_prompts)  # sorted for reproducibility given a fixed seed
eval_prompt_list = sorted(eval_prompts)

ppo_prompt_dataset = Dataset.from_dict({"prompt": [[{"role": "user", "content": p}] for p in ppo_prompt_list]})

print(f"PPO prompt dataset: {len(ppo_prompt_dataset):,} rows")
print(f"Eval holdout prompts: {len(eval_prompt_list):,}")
print(ppo_prompt_dataset[0])

# Train the reward model

Train a LoRA reward model (`AutoModelForSequenceClassification`, `num_labels=1`) on the harmlessness preference pairs using TRL's `RewardTrainer`.

In [ ]:
rm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **token_kwarg)
if rm_tokenizer.pad_token is None:
    rm_tokenizer.pad_token = rm_tokenizer.eos_token

rm_base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=1, dtype=TORCH_DTYPE, **token_kwarg
)

rm_peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    modules_to_save=["score"],  # train the reward head alongside the LoRA adapters
    task_type="SEQ_CLS",
)

rm_config = RewardConfig(
    output_dir=f"{RM_CHECKPOINT_PATH}_training",
    per_device_train_batch_size=RM_BATCH_SIZE,
    num_train_epochs=RM_NUM_EPOCHS,
    learning_rate=RM_LEARNING_RATE,
    max_length=RM_MAX_LENGTH,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=10,
    report_to="none",
    bf16=True,
)

reward_trainer = RewardTrainer(
    model=rm_base_model,
    args=rm_config,
    train_dataset=rm_train_dataset,
    eval_dataset=rm_eval_dataset,
    processing_class=rm_tokenizer,
    peft_config=rm_peft_config,
)

In [ ]:
reward_trainer.train()

## Visualize reward model training

In [ ]:
# Code in this block partially generated with Claude
rm_log_df = pd.DataFrame(reward_trainer.state.log_history)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

train_loss = rm_log_df.dropna(subset=["loss"]) if "loss" in rm_log_df else pd.DataFrame()
if not train_loss.empty:
    axes[0].plot(train_loss["step"], train_loss["loss"], color="#5c8ae0")
axes[0].set_title("Training Loss", fontsize=16)
axes[0].set_xlabel("Step")
axes[0].grid(linestyle="--", alpha=0.5)

eval_acc = rm_log_df.dropna(subset=["eval_accuracy"]) if "eval_accuracy" in rm_log_df else pd.DataFrame()
if not eval_acc.empty:
    axes[1].plot(eval_acc["step"], eval_acc["eval_accuracy"], color="#e05c5c", marker="o")
axes[1].axhline(0.5, linestyle=":", color="gray", label="Random chance")
axes[1].set_title("Eval Accuracy", fontsize=16)
axes[1].set_xlabel("Step")
axes[1].legend()
axes[1].grid(linestyle="--", alpha=0.5)

eval_margin = rm_log_df.dropna(subset=["eval_margin"]) if "eval_margin" in rm_log_df else pd.DataFrame()
if not eval_margin.empty:
    axes[2].plot(eval_margin["step"], eval_margin["eval_margin"], color="#5cbf6a", marker="o")
axes[2].set_title("Eval Margin (chosen - rejected reward)", fontsize=16)
axes[2].set_xlabel("Step")
axes[2].grid(linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

if not eval_acc.empty:
    final_acc = eval_acc["eval_accuracy"].iloc[-1]
    print(f"Final reward model eval accuracy: {final_acc:.1%}")
    if final_acc <= 0.5:
        print("WARNING: reward model accuracy is at or below random chance — investigate before proceeding to PPO.")

## Merge and save the reward model

In [ ]:
rm_merged_model = reward_trainer.model.merge_and_unload()
rm_merged_model.save_pretrained(RM_CHECKPOINT_PATH)
rm_tokenizer.save_pretrained(RM_CHECKPOINT_PATH)
print(f"Saved merged reward model to {RM_CHECKPOINT_PATH}")

# Free GPU memory before moving to PPO training
del reward_trainer, rm_base_model, rm_merged_model
report_gpu_memory("after reward model cleanup")

# PPO RLHF training

Run true reinforcement learning (PPO) on the policy model against the trained reward model, using TRL's `PPOTrainer`. The policy uses a LoRA adapter; the reference policy is the same weights with the adapter disabled (`ref_model=None`, handled automatically by `PPOTrainer` when `peft_config` is passed), so no separate full reference-model copy needs to be loaded.

## Load policy, reward model, and value model

In [ ]:
ppo_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left", **token_kwarg)
if ppo_tokenizer.pad_token is None:
    ppo_tokenizer.pad_token = ppo_tokenizer.eos_token

# Policy: the model being trained. LoRA adapter applied via `peft_config` passed to PPOTrainer below.
policy_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=TORCH_DTYPE, **token_kwarg)

# Reward model: frozen, judges (prompt, response) pairs. Loaded from the merged RM checkpoint.
reward_model = AutoModelForSequenceClassification.from_pretrained(
    RM_CHECKPOINT_PATH, num_labels=1, dtype=TORCH_DTYPE
)

# Value model: warm-started from the same reward model checkpoint, trained in full during PPO
# (TRL's PPOTrainer only LoRA-wraps the policy model, not the value model).
value_model = AutoModelForSequenceClassification.from_pretrained(
    RM_CHECKPOINT_PATH, num_labels=1, dtype=TORCH_DTYPE
)

ppo_peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type="CAUSAL_LM",
)

print("Policy, reward model, and value model loaded.")
report_gpu_memory("after loading PPO models")

## Prepare the PPO rollout dataset

Pre-tokenize prompts before training (TRL's PPO example pre-tokenizes and only collates during training).

In [ ]:
def prepare_ppo_dataset(dataset, tokenizer):
    """Pre-tokenize prompts (with chat template applied) before PPO training.

    `return_dict=True` is required here: Ministral 3's `mistral_common` tokenizer
    (`MistralCommonBackend`) always returns a `{"input_ids": ..., "attention_mask": ...}`
    dict from `apply_chat_template(tokenize=True)`, unlike standard fast tokenizers which
    return a plain list of ids by default. Explicitly requesting `return_dict=True` and
    indexing `["input_ids"]` gives a consistent plain list of ints either way.
    """

    def tokenize(element):
        output = tokenizer.apply_chat_template(
            element["prompt"], add_generation_prompt=True, tokenize=True, return_dict=True
        )
        input_ids = output["input_ids"]
        return {"input_ids": input_ids, "lengths": len(input_ids)}

    return dataset.map(tokenize, remove_columns=dataset.column_names)

ppo_train_dataset = prepare_ppo_dataset(ppo_prompt_dataset, ppo_tokenizer)
# Drop any overly long prompts so rollouts stay within a reasonable context budget
ppo_train_dataset = ppo_train_dataset.filter(lambda x: x["lengths"] <= PPO_MAX_PROMPT_LENGTH)
print(f"PPO train dataset (post-filter): {len(ppo_train_dataset):,} prompts")

## Configure and run PPOTrainer

In [ ]:
ppo_config = PPOConfig(
    output_dir=f"{FINAL_MODEL_PATH}_training",
    per_device_train_batch_size=PPO_PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=PPO_GRADIENT_ACCUMULATION_STEPS,
    num_mini_batches=PPO_MINI_BATCHES,
    total_episodes=len(ppo_train_dataset) * PPO_NUM_EPOCHS,
    num_ppo_epochs=PPO_NUM_EPOCHS,
    learning_rate=PPO_LEARNING_RATE,
    response_length=PPO_RESPONSE_LENGTH,
    kl_coef=PPO_KL_COEF,
    missing_eos_penalty=PPO_MISSING_EOS_PENALTY,
    stop_token="eos",
    temperature=EVAL_TEMPERATURE,
    local_rollout_forward_batch_size=PPO_PER_DEVICE_BATCH_SIZE,
    logging_steps=5,
    report_to="none",
    bf16=True,
)

ppo_trainer = PPOTrainer(
    args=ppo_config,
    processing_class=ppo_tokenizer,
    model=policy_model,
    ref_model=None,  # derived automatically from the LoRA-wrapped policy (adapters disabled)
    reward_model=reward_model,
    value_model=value_model,
    train_dataset=ppo_train_dataset,
    peft_config=ppo_peft_config,
)

In [ ]:
ppo_trainer.train()

## Visualize PPO training

Plot the key PPO metrics logged automatically to `trainer.state.log_history`: RLHF reward, KL divergence, raw reward-model scores, policy/value losses, and policy entropy.

In [ ]:
# Code in this block partially generated with Claude
ppo_log_df = pd.DataFrame(ppo_trainer.state.log_history)

metrics_to_plot = [
    ("objective/rlhf_reward", "RLHF Reward (score - KL penalty)", "#5cbf6a"),
    ("objective/scores", "Raw Reward Model Scores", "#5c8ae0"),
    ("objective/kl", "KL Divergence (policy vs. reference)", "#e0a05c"),
    ("loss/policy_avg", "Policy Loss", "#e05c5c"),
    ("loss/value_avg", "Value Loss", "#9b5ce0"),
    ("policy/entropy_avg", "Policy Entropy", "#5cbfbf"),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (col, title, color) in zip(axes.flat, metrics_to_plot):
    if col in ppo_log_df.columns:
        series = ppo_log_df.dropna(subset=[col])
        x = series["episode"] if "episode" in series.columns else series.index
        ax.plot(x, series[col], color=color)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Episode")
    ax.grid(linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

if "objective/rlhf_reward" in ppo_log_df.columns:
    reward_series = ppo_log_df.dropna(subset=["objective/rlhf_reward"])["objective/rlhf_reward"]
    if len(reward_series) >= 2:
        trend = "increased" if reward_series.iloc[-1] > reward_series.iloc[0] else "did not increase"
        print(f"objective/rlhf_reward {trend} over training: {reward_series.iloc[0]:.3f} -> {reward_series.iloc[-1]:.3f}")

## Merge and save the fine-tuned policy

Merge the PPO-trained LoRA adapter into the base weights and save the full model, so it can be loaded later with a plain `AutoModelForCausalLM.from_pretrained(FINAL_MODEL_PATH)` — no PEFT dependency needed downstream (e.g. in `rag_safety.ipynb`).

In [ ]:
# PPOTrainer wraps the policy inside a PolicyAndValueWrapper (`trainer.model.policy`).
# The policy is the LoRA-wrapped PeftModel since we passed `peft_config` above.
trained_policy = ppo_trainer.model.policy
final_merged_model = trained_policy.merge_and_unload()
final_merged_model.save_pretrained(FINAL_MODEL_PATH)
ppo_tokenizer.save_pretrained(FINAL_MODEL_PATH)
print(f"Saved merged fine-tuned model to {FINAL_MODEL_PATH}")

# Free GPU memory before the before/after evaluation phase
del ppo_trainer, policy_model, reward_model, value_model, trained_policy, final_merged_model
report_gpu_memory("after PPO cleanup")

# Before/after safety evaluation

Generate responses to the held-out eval prompts with (a) the original base model and (b) the fine-tuned model, then judge both response sets with Llama Guard 3 8B (same pattern as `rag_safety.ipynb`). Models are loaded and freed sequentially to keep peak GPU memory low.

## Generate "before" responses (original base model)

In [ ]:
@torch.no_grad()
def generate_responses(model, tokenizer, prompts: list[str], max_new_tokens: int, temperature: float) -> list[str]:
    """Generate one response per prompt using the model's chat template."""
    responses = []
    for prompt in tqdm(prompts, desc="Generating"):
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(DEVICE)
        gen_tokens = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )
        new_ids = gen_tokens[0][inputs["input_ids"].shape[-1]:]
        responses.append(tokenizer.decode(new_ids, skip_special_tokens=True).strip())
    return responses

eval_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **token_kwarg)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=TORCH_DTYPE, device_map="auto", **token_kwarg)

before_responses = generate_responses(
    base_model, eval_tokenizer, eval_prompt_list, EVAL_MAX_NEW_TOKENS, EVAL_TEMPERATURE
)

del base_model
report_gpu_memory("after base model cleanup")

## Generate "after" responses (fine-tuned model)

In [ ]:
finetuned_model = AutoModelForCausalLM.from_pretrained(FINAL_MODEL_PATH, dtype=TORCH_DTYPE, device_map="auto")

after_responses = generate_responses(
    finetuned_model, eval_tokenizer, eval_prompt_list, EVAL_MAX_NEW_TOKENS, EVAL_TEMPERATURE
)

del finetuned_model
report_gpu_memory("after fine-tuned model cleanup")

## Judge both response sets with Llama Guard 3 8B

In [ ]:
guard_tokenizer = AutoTokenizer.from_pretrained(LLAMA_GUARD_ID, **token_kwarg)
guard_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_GUARD_ID, dtype=TORCH_DTYPE, device_map="auto", **token_kwarg
)
guard_model.eval()

@torch.no_grad()
def judge_safety(user_message: str, assistant_response: str, max_new_tokens: int = 30) -> dict:
    """Judge whether a (user_message, assistant_response) pair is safe using Llama Guard 3."""
    conversation = [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": assistant_response},
    ]
    prompt = guard_tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    inputs = guard_tokenizer(prompt, return_tensors="pt").to(DEVICE)
    output_ids = guard_model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=guard_tokenizer.eos_token_id
    )
    new_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    raw = guard_tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    lines = raw.splitlines()
    verdict = lines[0].strip().lower() if lines else "unknown"
    category = lines[1].strip() if verdict == "unsafe" and len(lines) > 1 else None
    return {"verdict": verdict, "category": category}

before_verdicts = [judge_safety(p, r) for p, r in tqdm(zip(eval_prompt_list, before_responses), total=len(eval_prompt_list), desc="Judging before")]
after_verdicts = [judge_safety(p, r) for p, r in tqdm(zip(eval_prompt_list, after_responses), total=len(eval_prompt_list), desc="Judging after")]

eval_df = pd.DataFrame({
    "prompt": eval_prompt_list,
    "before_response": before_responses,
    "after_response": after_responses,
    "before_verdict": [v["verdict"] for v in before_verdicts],
    "before_category": [v["category"] for v in before_verdicts],
    "after_verdict": [v["verdict"] for v in after_verdicts],
    "after_category": [v["category"] for v in after_verdicts],
})

del guard_model
report_gpu_memory("after Llama Guard cleanup")

eval_df.to_csv(EVAL_RESULTS_PATH, index=False)
print(f"Saved before/after eval results to {EVAL_RESULTS_PATH}")
eval_df.head(3)

## Visualize before/after safety results

In [ ]:
# Code in this block partially generated with Claude
before_unsafe_rate = (eval_df["before_verdict"] == "unsafe").mean()
after_unsafe_rate = (eval_df["after_verdict"] == "unsafe").mean()
before_unsafe_n = (eval_df["before_verdict"] == "unsafe").sum()
after_unsafe_n = (eval_df["after_verdict"] == "unsafe").sum()

fig, ax = plt.subplots(figsize=(7, 6))
bars = ax.bar(
    ["Before RLHF\n(base model)", "After RLHF\n(fine-tuned)"],
    [before_unsafe_rate * 100, after_unsafe_rate * 100],
    color=["#e05c5c", "#5c8ae0"],
    zorder=3,
)
for bar, count in zip(bars, [before_unsafe_n, after_unsafe_n]):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
        f"{bar.get_height():.1f}%\nn = {int(count)}", ha="center", va="bottom", fontsize=13,
    )
ax.set_ylabel("Unsafe Response Rate (%)", fontsize=16)
ax.set_title("Unsafe Response Rate: Before vs. After Safety RLHF", fontsize=18)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
plt.tight_layout()
plt.show()

print(f"Before: {before_unsafe_rate:.1%} unsafe ({before_unsafe_n}/{len(eval_df)})")
print(f"After:  {after_unsafe_rate:.1%} unsafe ({after_unsafe_n}/{len(eval_df)})")
if after_unsafe_rate < before_unsafe_rate:
    print("Unsafe response rate decreased after PPO fine-tuning.")
else:
    print("Unsafe response rate did not decrease — treat as a finding, not necessarily a bug "
          "(safety gains from RLHF at this data/compute scale are not guaranteed).")

In [ ]:
# Category breakdown of unsafe responses (Llama Guard's MLCommons hazard taxonomy, S1-S14)
GUARD_CATEGORY_LABELS = {
    "S1": "Violent Crimes", "S2": "Non-Violent Crimes", "S3": "Sex-Related Crimes",
    "S4": "Child Sexual Exploitation", "S5": "Defamation", "S6": "Specialized Advice",
    "S7": "Privacy", "S8": "Intellectual Property", "S9": "Indiscriminate Weapons",
    "S10": "Hate", "S11": "Suicide & Self-Harm", "S12": "Sexual Content",
    "S13": "Elections", "S14": "Code Interpreter Abuse",
}
ALL_GUARD_CATEGORIES = [f"S{i}" for i in range(1, 15)]

before_unsafe_cats = eval_df.loc[eval_df["before_verdict"] == "unsafe", "before_category"].value_counts()
after_unsafe_cats = eval_df.loc[eval_df["after_verdict"] == "unsafe", "after_category"].value_counts()

before_counts = [before_unsafe_cats.get(cat, 0) for cat in ALL_GUARD_CATEGORIES]
after_counts = [after_unsafe_cats.get(cat, 0) for cat in ALL_GUARD_CATEGORIES]

x = np.arange(len(ALL_GUARD_CATEGORIES))
width = 0.38

fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(x - width / 2, before_counts, width, label="Before RLHF", color="#e05c5c", zorder=3)
ax.bar(x + width / 2, after_counts, width, label="After RLHF", color="#5c8ae0", zorder=3)

tick_labels = [f"S{i}\n{GUARD_CATEGORY_LABELS[f'S{i}']}" for i in range(1, 15)]
ax.set_xticks(x)
ax.set_xticklabels(tick_labels, rotation=40, ha="right", fontsize=11)
ax.set_xlabel("Safety Category", fontsize=16)
ax.set_ylabel("Unsafe Response Count", fontsize=16)
ax.set_title("Unsafe Responses by Category — Before vs. After Safety RLHF", fontsize=18)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.legend(fontsize=13)
ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
plt.tight_layout()
plt.show()

# Summary and next steps

The safety-RLHF fine-tuned model is saved at `FINAL_MODEL_PATH` (`{DRIVE_ROOT}/ministral_3b_safety_rlhf`) as a full-precision, merged checkpoint — no PEFT/LoRA dependency is needed to load it.

**To use this model in `rag_safety.ipynb` in place of Command-R:**

- Replace `COMMAND_R_ID` with the path to `FINAL_MODEL_PATH` and load with `AutoModelForCausalLM`/`AutoTokenizer` as usual.
- `generate_with_rag` currently calls Command-R's Cohere-specific `apply_grounded_generation_template`, which Ministral 3 does not support. This needs to be replaced with a generic prompt template that manually inserts the retrieved documents into the user turn (e.g. `"Context:\n{documents}\n\nQuestion: {query}"`) before applying Ministral's standard chat template.
- `generate_without_rag` should work largely unchanged, since it only relies on the standard `apply_chat_template` path.
- Llama Guard judging and the FAISS/BGE-M3 retrieval pipeline require no changes.

This adaptation is out of scope for this notebook and is flagged here as a follow-up task.